# Medical Document Analysis with Strands Agent

This notebook demonstrates how to use Strands Agent to:
1. Extract text from PDF medical documents
2. Infer key medical information (diagnosis, medications, treatments)
3. Enrich the data with standardized medical codes (ICD-10, RxNorm, SNOMED CT)

## Setup and Dependencies


### Prerequisites
* Python 3.10+
* AWS account
* Anthropic Claude 3.7 enabled on Amazon Bedrock, [guide](https://docs.aws.amazon.com/bedrock/latest/userguide/model-access.html)
* IAM role with permissions to create Amazon Bedrock Knowledge Base, Amazon S3 bucket

Let's now install the requirement packages for our Strands Agent

In [1]:
# installing pre-requisites
!pip install -r requirements.txt

  Using cached strands_agents-0.1.6-py3-none-any.whl.metadata (10 kB)
  Using cached strands_agents_tools-0.1.4-py3-none-any.whl.metadata (20 kB)
  Using cached pypdf-5.6.0-py3-none-any.whl.metadata (7.2 kB)
  Using cached reportlab-4.4.1-py3-none-any.whl.metadata (1.8 kB)
  Using cached docstring_parser-0.15-py3-none-any.whl.metadata (2.4 kB)
  Using cached watchdog-6.0.0-py3-none-manylinux2014_x86_64.whl.metadata (44 kB)
  Using cached aws_requests_auth-0.4.3-py2.py3-none-any.whl.metadata (567 bytes)
  Using cached dill-0.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached pillow-11.2.1-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (8.9 kB)
  Using cached prompt_toolkit-3.0.51-py3-none-any.whl.metadata (6.4 kB)
  Using cached PyJWT-2.10.1-py3-none-any.whl.metadata (4.0 kB)
  Using cached rich-14.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached slack_bolt-1.23.0-py2.py3-none-any.whl.metadata (11 kB)
  Using cached tenacity-9.1.2-py3-none-any.whl.metadata (1.2 kB)
  Using cached 

In [2]:
pip install --upgrade strands-agents


Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install strands-agents-tools strands-agents-builder

  Using cached strands_agents_builder-0.1.2-py3-none-any.whl.metadata (11 kB)
  Using cached halo-0.0.31-py3-none-any.whl
  Using cached log_symbols-0.0.14-py3-none-any.whl.metadata (523 bytes)
  Using cached spinners-0.0.24-py3-none-any.whl.metadata (576 bytes)
  Using cached ollama-0.5.1-py3-none-any.whl.metadata (4.3 kB)
  Using cached pydantic-2.11.5-py3-none-any.whl.metadata (67 kB)
  Using cached pydantic_core-2.33.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.8 kB)
Using cached strands_agents_builder-0.1.2-py3-none-any.whl (27 kB)
Using cached log_symbols-0.0.14-py3-none-any.whl (3.1 kB)
Using cached ollama-0.5.1-py3-none-any.whl (13 kB)
Using cached pydantic-2.11.5-py3-none-any.whl (444 kB)
Using cached pydantic_core-2.33.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (2.0 MB)
Using cached spinners-0.0.24-py3-none-any.whl (5.5 kB)
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.18.4
    Uninstalling

In [4]:

!pip show strands-agents-tools

Name: strands-agents-tools
Version: 0.1.4
Summary: A collection of specialized tools for Strands Agents
Home-page: https://github.com/strands-agents/tools
Author: 
Author-email: AWS <opensource@amazon.com>
License: Apache-2.0
Location: /opt/conda/lib/python3.11/site-packages
Requires: aws-requests-auth, colorama, dill, pillow, prompt-toolkit, pyjwt, rich, slack-bolt, strands-agents, sympy, tenacity, watchdog
Required-by: strands-agents-builder


In [5]:
!pip show strands-agents

Name: strands-agents
Version: 0.1.6
Summary: A model-driven approach to building AI agents in just a few lines of code
Home-page: https://github.com/strands-agents/sdk-python
Author: 
Author-email: AWS <opensource@amazon.com>
License: Apache-2.0
Location: /opt/conda/lib/python3.11/site-packages
Requires: boto3, botocore, docstring-parser, mcp, opentelemetry-api, opentelemetry-exporter-otlp-proto-http, opentelemetry-sdk, pydantic, typing-extensions, watchdog
Required-by: strands-agents-builder, strands-agents-tools


# Define Medical Agent
The Medical Document Processing Assistant is an AI-powered tool designed to extract, analyze, and enrich medical information from various document formats such as PDFs and images. This assistant specializes in processing clinical notes, pathology reports, discharge summaries, and other medical documents to provide structured data with standardized medical coding. We will be using Strands to define the agent. 

![arch](architecture.png)

We will defining tools to determine and infer ICD10 code, RxNorm and SNOMED. We are calling the following API within the tool to make the determination. 

1. ICD-10-CM & ICD-10-PCS (U.S. Versions)
Official Source: U.S. Centers for Medicare & Medicaid Services (CMS) and the National Center for Health Statistics (NCHS)

ICD-10-CM (diagnoses):

Call API: https://clinicaltables.nlm.nih.gov/apidoc/icd10cm/v3/doc.html


ICD-10-PCS (procedures):

https://www.cms.gov/medicare/icd-10/2025-icd-10-pcs
(Adjust year as needed)

2. RxNorm
Official Source: U.S. National Library of Medicine (NLM)

https://lhncbc.nlm.nih.gov/RxNav/APIs/RxNormAPIs.html?_gl=1*1qdlo6u*_ga*ODQ1ODkzMzMyLjE3NDg4MzYwMjc.*_ga_7147EPK006*czE3NDg4MzYwMjYkbzEkZzEkdDE3NDg4MzY2MDAkajYwJGwwJGgw*_ga_P1FPTH9PL4*czE3NDg4MzYwMjYkbzEkZzEkdDE3NDg4MzY2MDAkajYwJGwwJGgw

4. SNOMED CT
International Edition

Official Source: SNOMED 

https://browser.ihtsdotools.org/?perspective=full&conceptId1=404684003&edition=MAIN/SNOMEDCT-US/2025-03-01&release=&languages=en



In [6]:
import os
import json
from typing import Dict, List, Optional, Any
import pypdf
import boto3
from strands import Agent, tool
from strands.models import BedrockModel

In [14]:
import os
import logging
from strands import Agent
from strands_tools import file_read
from document_processor import process_document
from medical_coding_tools import (
    get_icd, get_rx, get_snomed,
    link_icd, link_rx, link_snomed
)
# System prompt for the medical document processing agent
SYSTEM_PROMPT = """
You are a Medical Document Processing Assistant specialized in extracting and analyzing medical information from clinical documents.

Your tasks include:
1. Processing medical documents (PDFs, images) to extract text
2. Identifying key medical information: diagnoses, medications, treatments
3. Enriching the extracted information with standardized medical codes:
   - ICD-10 codes for diagnoses
   - RxNorm codes for medications
   - SNOMED CT codes for treatments

Provide clear, accurate, and structured information that can be used by healthcare professionals.
"""

# Create the medical document processing agent
medical_agent = Agent(
    system_prompt=SYSTEM_PROMPT,
    tools=[
        file_read,
        process_document,
        get_icd,
        get_rx,
        get_snomed,
        link_icd,
        link_rx,
        link_snomed
    ]
)


# Testing

In [15]:
# Extracted data

# Example clinical note
CLINICAL_NOTE = """
Carlie had a seizure 2 weeks ago. She is complaining of frequent headaches
Nausea is also present. She also complains of eye trouble with blurry vision
Meds : Topamax 50 mgs at breakfast daily,
Send referral order to neurologist
Follow-up as scheduled
"""
# Process the clinical note
response = medical_agent(
        f"Process this clinical note and extract diagnoses, medications, and treatments with their respective medical codes: {CLINICAL_NOTE}"
    )   
print("\nProcessing complete!\n")
print("Usage metrics:")
print(medical_agent.event_loop_metrics)


I'll process this clinical note to extract diagnoses, medications, and treatments with their respective medical codes. Let me break this down step by step.

First, let me analyze the clinical text to extract diagnoses and their ICD-10 codes:
Tool #8: link_icd
Now let me extract medications and their RxNorm codes:
Tool #9: link_rx
Finally, let me extract treatments/procedures and their SNOMED CT codes:
Tool #10: link_snomed
## Clinical Note Analysis: Summary Report

### Diagnoses (with ICD-10 Codes):
1. **Seizure**
   - ICD-10: R56.9 (Convulsions, not elsewhere classified)
2. **Headaches**
   - ICD-10: R51 (Headache)
3. **Nausea**
   - ICD-10: R11 (Nausea and vomiting)
4. **Blurry Vision**
   - ICD-10: H53.8 (Other visual disturbances)

### Medications (with RxNorm Codes):
1. **Topamax**
   - Dosage: 50 mg
   - Frequency: at breakfast daily
   - RxNorm Code: 309362
   - RxNorm Name: topiramate 50 MG Oral Tablet

### Treatments/Procedures (with SNOMED CT Codes):
1. **Medication Administr

In [16]:
# PDF medical document as an input
file_path='p1.pdf'
response = medical_agent(f"Process this medical document and extract diagnoses, medications, and treatments with their respective medical codes: {file_path}")

print("\nProcessing complete!\n")
print("Usage metrics:")
print(medical_agent.event_loop_metrics)

I'll process the medical document "p1.pdf" to extract diagnoses, medications, and treatments with their respective medical codes. Let me first process the document to extract its content.
Tool #11: process_document
Now, I'll extract the diagnoses from the clinical text and link them to ICD-10 codes:
Tool #12: link_icd
Let me get more specific information for leiomyosarcoma:
Tool #13: get_icd
Let me try another search:
Tool #14: get_icd
Let me search for relevant SNOMED CT codes for treatments:
Tool #15: link_snomed
Based on the processed document, here is my comprehensive analysis:

## Medical Document Analysis: Pathology Report Summary

### Patient Information:
- **Name**: MECTOMY, MYA
- **MRN**: R000011230
- **DOB**: 04/23/1970 (46 years, Female)
- **Procedure Date**: 07/08/2016
- **Report Date**: 07/12/2016
- **Surgeon/Attending**: RAYMOND E CLARKE, MD

### Primary Diagnosis (with ICD-10 Code):
- **Leiomyosarcoma of Uterus**
  - ICD-10: C54.1 (Malignant neoplasm of corpus uteri)
  -

## Conclusion

This notebook demonstrates how to use Strands Agent to:
1. Extract text from PDF medical documents
2. Identify key medical information (diagnoses, medications, treatments)
3. Enrich the data with standardized medical codes (ICD-10, RxNorm, SNOMED CT)

The agent uses a combination of tools to perform these tasks:
- PDF text extraction
- Medical code lookup (ICD-10, RxNorm, SNOMED CT)
- Medical information enrichment

This approach can be extended to handle more complex medical documents and integrate with real medical code databases or APIs.